# CTAug augmentation notebook

Runs each CTAug augmentation on a single cardiac CT and writes the result next to the
original as NIfTI, so the simulated artifacts can be inspected in a viewer such as
ITK-SNAP.

Every call produces, under `artifact_results/<scan-name>/`:

- `aug_<TransformName>_img.nii.gz` — the augmented image
- `aug_<TransformName>_seg.nii.gz` — the shifted labels (step/motion transforms only, since
  those move anatomy and the segmentation has to follow)
- `org_img.nii.gz` / `org_seg.nii.gz` — the exact input the transform saw, for a side-by-side
- `<TransformName>_info.json` — every parameter that was sampled, which is enough to replay it

The README's *Examples on a real scan* section walks through the `info` of each of these
outputs and shows how to convert its millimetre values into cursor coordinates.

In [ ]:
# Install CTAug plus the I/O helpers this notebook uses for NIfTI and JSON.
!pip install -q ctaug deep_utils itk simpleITK

In [ ]:
# imports
import os
import random
from os.path import join, split

import numpy as np
from deep_utils import SITKUtils, JsonUtils

from ctaug import (
    CalcificationTransform,
    MetalTransform,
    MotionTransform,
    StepMotionTransform,
    StepTransform,
    WireTransform,
)
from ctaug.metal.functional import simulate_artifacts_unified

In [ ]:
# Fixed seed so repeated runs land the artifacts in the same place.
# Comment it out to draw a new position/severity/intensity on every run.
random.seed(1234)

In [ ]:
# Input scan and output directory. Paths are relative to this notebook, which lives
# in `example/`; the segmentation is used to place artifacts in plausible anatomy.
img_path = "10151662.nii.gz"
seg_path = "10151662_seg.nii.gz"
output_path = "artifact_results"

# Artifacts — calcification, wire, and metal

These three transforms burn a high-HU object into the volume and then simulate the streaks
it would cause: each affected slice is forward-projected (Radon), the sinogram rows explained
by the object are dampened by `severity`, and the slice is reconstructed (inverse Radon).

Two consequences worth knowing before reading the results:

- They need `spacing` in **`(z, y, x)`** order, because their geometry is sampled in
  millimetres. SimpleITK reports spacing as `(x, y, z)`, hence the `[::-1]` below.
- The nominal `intensity` in `info` is *not* what you measure in the output. The projection
  round trip spreads the object out, and the transform clips its result back to the input
  volume's original min/max.

Placement is anchored on the segmentation: `include_labels=(2, 3, 4)` restricts it to those
labels (pass `exclude_labels` instead to allow everything but a few). `crop=(128, 256, 256)`
takes a centre crop first — the projection round trip is the expensive part, so cropping keeps
the runtime sane.

Any other transform argument — `intensity`, `severity`, `max_n_specs` — can be passed straight
through the helper below; `intensity` and `severity` each take a scalar or a `(low, high)` range.

Each writes its parameters to its own `<TransformName>_info.json`, so the runs below don't
overwrite each other and any of them can be replayed later.

In [ ]:
def easy_metal_aug(img_path, seg_path, output_path, info_path=None,
                   crop=None, exclude_labels=None,
                   include_labels=(2, 3, 4),
                   exclude_class=None,
                   augmentation_class=MetalTransform,
                   **cls_kwargs):
    """Insert a calcification/wire/metal implant and save image, original and info.

    Extra keyword arguments (``intensity``, ``severity``, ``max_n_specs``, ...) go straight
    to the transform.
    """
    img_name = split(img_path)[-1].replace("_0000", "").replace(".nii.gz", "")
    # create output dir
    os.makedirs(join(output_path, img_name), exist_ok=True)
    # read image and seg files
    img_arr, img_img = SITKUtils.get_array_img(img_path)
    seg_arr, seg_img = SITKUtils.get_array_img(seg_path)
    print(f"[INFO] img: {img_arr.shape} and seg: {seg_arr.shape}")

    if crop is not None:
        # centre crop to `crop`, keeping image and segmentation aligned
        exact = crop
        z_dis = img_arr.shape[0] - exact[0]
        z_s = z_dis // 2
        z_e = z_s + exact[0]

        y_dis = img_arr.shape[1] - exact[1]
        y_s = y_dis // 2
        y_e = y_s + exact[1]

        x_dis = img_arr.shape[2] - exact[2]
        x_s = x_dis // 2
        x_e = x_s + exact[2]

        chosen_arr = img_arr[z_s: z_e, y_s:y_e, x_s: x_e]
        chosen_seg = seg_arr[z_s: z_e, y_s:y_e, x_s: x_e]

        img_arr = chosen_arr
        seg_arr = chosen_seg

    # If you want to try from previous settings: replay a saved info.json instead of
    # sampling new parameters. Handy for reproducing a specific artifact on another scan.
    if info_path:
        info = JsonUtils.load(info_path)
        augmented_array = simulate_artifacts_unified(img_arr, angles=np.linspace(0, 180, info['angle_size'], endpoint=False),
                                                     implant_specs=info["implant_specs"], severity=info["severity"],
                                                     spacing=info['spacing'])
        augmented_array = np.clip(augmented_array, a_min=img_arr.min(), a_max=img_arr.max())
        # replayed volumes are tagged "from_info" so they never overwrite a sampled one
        augmentor_class_name = "from_info"
    else:
        # CTAug wants spacing as (z, y, x); SimpleITK gives (x, y, z).
        if exclude_class is not None:
            augmentor = augmentation_class(spacing=img_img.GetSpacing()[::-1],
                                           exclude_labels=exclude_labels,
                                           include_labels=include_labels,
                                           exclude_class=exclude_class,
                                           verbose=True, **cls_kwargs)
        else:
            augmentor = augmentation_class(spacing=img_img.GetSpacing()[::-1], exclude_labels=exclude_labels, include_labels=include_labels,
                                           verbose=True, **cls_kwargs)
        # transforms are dict-in/dict-out; add the channel axis they expect
        aug_output = augmentor(image=img_arr[None, ...], segmentation=seg_arr[None, ...])
        augmented_array = aug_output['image'][0]
        # sampled parameters are returned under "<ClassName>_info"
        augmentor_class_name = augmentor.__class__.__name__
        info = aug_output[f"{augmentor_class_name}_info"]
        # one info file per transform, so runs don't overwrite each other
        JsonUtils.dump_safe_numpy(f"{output_path}/{img_name}/{augmentor_class_name}_info.json", info)

    aug_img_output_path = f"{output_path}/{img_name}/aug_{augmentor_class_name}_img.nii.gz"
    print(f"[INFO] Augment output: {aug_img_output_path}, info:\n{info}")
    SITKUtils.save_sample(aug_img_output_path, augmented_array, img=img_img)
    SITKUtils.save_sample(f"{output_path}/{img_name}/org_img.nii.gz", img_arr, img=img_img)
    SITKUtils.save_sample(f"{output_path}/{img_name}/org_seg.nii.gz", seg_arr, img=seg_img)

## Calcification

Small 3D ellipsoids, a few millimetres across. Their `info` reports `center_mm` and
`radius_mm` in **millimetres** — divide by `spacing` to get an index you can type into a
viewer's cursor box.

In [ ]:
# CalcificationTransform
easy_metal_aug(img_path,
               seg_path,
               output_path,
               crop=(128, 256, 256),
               exclude_labels=None,
               include_labels=(2, 3, 4),
               exclude_class=None,
               augmentation_class=CalcificationTransform,
               )

## Wire

A curved, high-HU lead (e.g. a pacemaker wire). Note that its `center_mm` is the arc's
**centre of curvature**, not a point on the wire — the wire itself sits `arc_radius_mm` away,
and `length` is the number of points sampled along the arc rather than a physical length.

In [ ]:
# WireTransform
easy_metal_aug(img_path,
               seg_path,
               output_path,
               crop=(128, 256, 256),
               exclude_labels=None,
               include_labels=(2, 3, 4),
               exclude_class=None,
               augmentation_class=WireTransform,
               )

## Metal

2D implants (circle / ellipse / rectangle) repeated over a run of slices. These are the
easiest to locate: `slices` holds the literal z indices and `center_px` is already in pixels,
ordered `(row, column)`.

This call also shows the pass-through kwargs: `intensity=(7000, 8000)` and `severity=(0.9, 0.99)`
push the implant well past the `(1000, 5000)` / `(0.1, 0.9)` defaults, for a deliberately obvious
artifact.

In [ ]:
# MetalTransform
easy_metal_aug(img_path,
               seg_path,
               output_path,
               crop=(128, 256, 256),
               exclude_labels=None,
               include_labels=(2, 3, 4),
               exclude_class=None,
               augmentation_class=MetalTransform,
               # both accept a scalar or a (low, high) range, sampled per implant
               intensity=(7000, 8000), severity=(0.9, 0.99),
               )

# Step-and-shoot artifacts

Three transforms, run on the **full** volume (no crop — they are cheap, no projection
involved):

- `StepTransform` — an intensity step across one axis
- `MotionTransform` — a block of the volume translated along another axis
- `StepMotionTransform` — both at the same boundary

**`StepMotionTransform` is the variant used in the paper**, where it is reported as
"step-and-shoot" and shortened to "step" for simplicity. The other two are its halves,
exposed separately for finer-grained control.

Unlike the artifact transforms above, these move anatomy, so the segmentation is shifted by
the same amount and saved alongside the image.

In [ ]:
def easy_step_aug(img_path, seg_path, org_output_path, augmentation_class=StepTransform, **cls_kwargs):
    """Apply a step/motion transform and save image, shifted segmentation and info."""
    img_name = split(img_path)[-1].replace("_0000", "").replace(".nii.gz", "")
    # create output dir
    os.makedirs(join(org_output_path, img_name), exist_ok=True)
    # read image and seg files
    img_arr, img_img = SITKUtils.get_array_img(img_path)
    seg_arr, seg_img = SITKUtils.get_array_img(seg_path) if seg_path else (None, None)
    print(f"[INFO] img: {img_arr.shape} and seg: {None if seg_arr is None else seg_arr.shape}")

    # any extra kwargs (cut_off_pixel_value_weight, motion_move_range, ...) go to the transform
    augmentor = augmentation_class(**cls_kwargs)

    # These transforms take a plain (Z, Y, X) numpy array and return the same container type,
    # dtype and shape, so there is no channel axis or torch conversion to do here.
    # MotionTransform requires a segmentation; the other two treat it as optional.
    try:
        aug_output = augmentor(image=img_arr, segmentation=seg_arr)
        seg_augmented_array = aug_output['segmentation']
    except Exception:
        aug_output = augmentor(image=img_arr)
        seg_augmented_array = None

    augmented_array = aug_output['image']
    cls_name = augmentor.__class__.__name__
    info = aug_output[f'{cls_name}_info']

    out_dir = f"{org_output_path}/{img_name}"
    # one info file per transform, so runs don't overwrite each other
    JsonUtils.dump_safe_numpy(f"{out_dir}/{cls_name}_info.json", info)

    aug_img_output_path = f"{out_dir}/aug_{cls_name}_img.nii.gz"
    aug_seg_output_path = f"{out_dir}/aug_{cls_name}_seg.nii.gz"
    print(f"[INFO] Augment output: {aug_img_output_path}, info:\n{info}")
    SITKUtils.save_sample(aug_img_output_path, augmented_array, img=img_img)
    SITKUtils.save_sample(f"{out_dir}/org_img.nii.gz", img_arr, img=img_img)
    if seg_augmented_array is not None:
        SITKUtils.save_sample(aug_seg_output_path, seg_augmented_array, img=seg_img)
        SITKUtils.save_sample(f"{out_dir}/org_seg.nii.gz", seg_arr, img=seg_img)

## Step

`cut_off_pixel_value_weight` scales the offset relative to the volume's mean intensity, so
`(0.1, 1)` is a considerably harsher step than the `(0.2, 0.6)` default.

In [ ]:
# StepTransform -- intensity step only
easy_step_aug(img_path, seg_path, output_path,
              augmentation_class=StepTransform,
              cut_off_pixel_value_weight=(0.1, 1),
              )

## Motion

`motion_move_range` is a fraction of the moved axis' extent, so `(0.01, 0.1)` shifts the block
by 1–10% of that axis.

In [ ]:
# MotionTransform -- spatial shift only (needs a segmentation)
easy_step_aug(img_path, seg_path, output_path,
              augmentation_class=MotionTransform,
              motion_move_range=(0.01, 0.1),
              )

## Step + motion

The combination used in the paper: an intensity step *and* a shift at the same boundary, which
is what a real gated-acquisition seam looks like.

In [ ]:
# StepMotionTransform -- the variant used in the paper
easy_step_aug(img_path, seg_path, output_path,
              augmentation_class=StepMotionTransform,
              cut_off_pixel_value_weight=(0.1, 1),
              motion_move_range=(0.01, 0.1),
              )

# Replaying a saved augmentation

Passing `info_path` skips the sampling and rebuilds the artifact from a saved
`<TransformName>_info.json` — useful for putting the exact same implant on a different scan, for
regenerating a figure, or for hand-editing the parameters. Only the artifact transforms support
this; the step transforms sample too little to be worth replaying.

Replayed volumes are written as `aug_from_info_img.nii.gz` rather than under the transform's
name, so they never overwrite a sampled run.

Editing a copy of the file is the way to push an artifact past what the transform would draw —
e.g. moving a single implant a few voxels or reshaping it. To simply widen the sampled ranges you
do not need a replay at all — pass `intensity=` and `severity=` to `easy_metal_aug`, as the metal
cell does.

In [ ]:
# Replay saved artifact settings instead of drawing new ones. Point `info_path` at the file
# written by the metal cell above, or at an edited copy of it.
easy_metal_aug(img_path,
               seg_path,
               output_path,
               info_path=join(output_path, "10151662", "MetalTransform_info.json"),
               crop=(128, 256, 256),
               )